In [ ]:
# import torch
# print(torch.__version__)

# # PGNN forward desing,,,,,, R**2 vs number of data


# =========================================
# 🔥 FULL INVERSE DESIGN LEARNING CURVE
# =========================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# =========================================
# 1️⃣ Load dataset
# =========================================
file_path = '/content/sample_data/data 2.xlsx'
data = pd.read_excel(file_path)
data.columns = data.columns.str.strip().str.lower()

# Inputs (energy)
e1 = data[['e1']].values.astype('float32')
e2 = data[['e2']].values.astype('float32')
e3 = data[['e3']].values.astype('float32')

# Outputs (thickness)
t1 = data[['t1']].values.astype('float32')
t2 = data[['t2']].values.astype('float32')
t3 = data[['t3']].values.astype('float32')

# =========================================
# 2️⃣ Neural Network
# =========================================
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1,64),
            nn.ReLU(),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Linear(64,1)
        )

    def forward(self,x):
        return self.net(x)

# =========================================
# 3️⃣ Learning Curve
# =========================================
max_samples = min(50, len(X))  # ensures no crash if dataset < 2000
subset_sizes = np.linspace(10, max_samples, 50, dtype=int)
r2_values = []

epochs = 500

for n in subset_sizes:

    # Subset
    e1_sub, e2_sub, e3_sub = e1[:n], e2[:n], e3[:n]
    t1_sub, t2_sub, t3_sub = t1[:n], t2[:n], t3[:n]

    # -------------------------------
    # Normalize (per subset)
    # -------------------------------
    scaler_e = StandardScaler()
    scaler_t = StandardScaler()

    e_all = np.vstack([e1_sub, e2_sub, e3_sub])
    t_all = np.vstack([t1_sub, t2_sub, t3_sub])

    scaler_e.fit(e_all)
    scaler_t.fit(t_all)

    e1_s = scaler_e.transform(e1_sub)
    e2_s = scaler_e.transform(e2_sub)
    e3_s = scaler_e.transform(e3_sub)

    t1_s = scaler_t.transform(t1_sub)
    t2_s = scaler_t.transform(t2_sub)
    t3_s = scaler_t.transform(t3_sub)

    # Tensors
    e1_t = torch.tensor(e1_s, dtype=torch.float32)
    e2_t = torch.tensor(e2_s, dtype=torch.float32)
    e3_t = torch.tensor(e3_s, dtype=torch.float32)

    t1_t = torch.tensor(t1_s, dtype=torch.float32)
    t2_t = torch.tensor(t2_s, dtype=torch.float32)
    t3_t = torch.tensor(t3_s, dtype=torch.float32)

    # -------------------------------
    # Models
    # -------------------------------
    model1 = Net()
    model2 = Net()
    model3 = Net()

    opt1 = optim.Adam(model1.parameters(), lr=1e-3)
    opt2 = optim.Adam(model2.parameters(), lr=1e-3)
    opt3 = optim.Adam(model3.parameters(), lr=1e-3)

    loss_fn = nn.MSELoss()

    # -------------------------------
    # Training
    # -------------------------------
    for epoch in range(epochs):

        # t1
        opt1.zero_grad()
        pred1 = model1(e1_t)
        loss1 = loss_fn(pred1, t1_t)
        loss1.backward()
        opt1.step()

        # t2
        opt2.zero_grad()
        pred2 = model2(e2_t)
        loss2 = loss_fn(pred2, t2_t)
        loss2.backward()
        opt2.step()

        # t3
        opt3.zero_grad()
        pred3 = model3(e3_t)
        loss3 = loss_fn(pred3, t3_t)
        loss3.backward()
        opt3.step()

    # -------------------------------
    # Prediction
    # -------------------------------
    with torch.no_grad():
        t1_pred = scaler_t.inverse_transform(model1(e1_t).numpy())
        t2_pred = scaler_t.inverse_transform(model2(e2_t).numpy())
        t3_pred = scaler_t.inverse_transform(model3(e3_t).numpy())

    # Combine all layers
    t_real = np.concatenate([t1_sub, t2_sub, t3_sub])
    t_pred = np.concatenate([t1_pred, t2_pred, t3_pred])

    # R²
    r2 = r2_score(t_real, t_pred)
    r2_values.append(r2)

# =========================================
# 4️⃣ Plot Learning Curve
# =========================================
plt.figure(figsize=(7,5))
plt.plot(subset_sizes, r2_values, marker='o')
plt.xlabel("Number of Samples")
plt.ylabel("R² (Thickness Prediction)")
plt.title("Inverse Design: R² vs Data Size")
plt.grid(True)

# =========================================
# 5️⃣ Convergence Detection
# =========================================
delta_r2 = np.diff(r2_values)

r2_threshold = 0.99
smooth_threshold = 0.001
window = 3

converged_sample = None

for i in range(len(delta_r2) - window):

    if (r2_values[i] >= r2_threshold and
        np.all(np.abs(delta_r2[i:i+window]) < smooth_threshold)):

        converged_sample = subset_sizes[i]
        print(f"\n✅ Converged at ~{converged_sample} samples")
        break

if converged_sample is not None:
    plt.axvline(converged_sample, linestyle='--', label='Convergence')
    plt.legend()

plt.show()


In [ ]:
# ########scatter extrapolation forward design graphs (e-t) repot 14

# =========================================
# Physics-Guided Neural Network (UPDATED)
# E = 0.4 * exp(0.95 * t)
# =========================================

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# -------------------------------
# 1️⃣ Load dataset
# -------------------------------
file_path = '/content/sample_data/data_new.xlsx'
data = pd.read_excel(file_path)

data.columns = data.columns.str.strip().str.lower().str.replace(' ', '_')

# -------------------------------
# 2️⃣ Extract inputs / outputs
# -------------------------------
X = data[['t1','t2','t3']].values.astype('float32')
y = data[['e1','e2','e3','e_total','m_total']].values.astype('float32')

# -------------------------------
# 3️⃣ Normalize
# -------------------------------
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)

X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y_scaled, dtype=torch.float32)

y_scale = torch.tensor(scaler_y.scale_, dtype=torch.float32)
y_mean  = torch.tensor(scaler_y.mean_, dtype=torch.float32)

# -------------------------------
# 4️⃣ Neural Network
# -------------------------------
class PGNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 5)
        )

    def forward(self, x):
        return self.net(x)

model = PGNN()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# -------------------------------
# 5️⃣ Physics equation (UPDATED)
# -------------------------------
def row_energy(t):
    return 0.3 * torch.exp(1.216 * t)

# -------------------------------
# 6️⃣ R² vs Dataset Size
# -------------------------------
physics_weight = 1  # 🔥 reduced for exponential stability
data_weight = 1

sizes = np.linspace(10, len(X), 50, dtype=int)
r2_list = []

for size in sizes:
    X_sub = X[:size]
    y_sub = y[:size]

    scaler_X_sub = StandardScaler()
    scaler_y_sub = StandardScaler()

    X_scaled_sub = scaler_X_sub.fit_transform(X_sub)
    y_scaled_sub = scaler_y_sub.fit_transform(y_sub)

    X_tensor_sub = torch.tensor(X_scaled_sub, dtype=torch.float32)
    y_tensor_sub = torch.tensor(y_scaled_sub, dtype=torch.float32)

    y_scale_sub = torch.tensor(scaler_y_sub.scale_, dtype=torch.float32)
    y_mean_sub  = torch.tensor(scaler_y_sub.mean_, dtype=torch.float32)

    model_sub = PGNN()
    optimizer_sub = optim.Adam(model_sub.parameters(), lr=1e-3)

    for epoch in range(1000):
        optimizer_sub.zero_grad()

        y_pred_scaled = model_sub(X_tensor_sub)
        y_pred = y_pred_scaled * y_scale_sub + y_mean_sub

        t_real = torch.tensor(X_sub, dtype=torch.float32)
        t1, t2, t3 = t_real[:,0], t_real[:,1], t_real[:,2]

        E1_pred, E2_pred, E3_pred = y_pred[:,0], y_pred[:,1], y_pred[:,2]

        physics_loss = ((E1_pred - row_energy(t1))**2 +
                        (E2_pred - row_energy(t2))**2 +
                        (E3_pred - row_energy(t3))**2).mean()

        data_loss = loss_fn(y_pred_scaled, y_tensor_sub)

        loss = data_weight * data_loss + physics_weight * physics_loss
        loss.backward()
        optimizer_sub.step()

    with torch.no_grad():
        y_pred_scaled = model_sub(X_tensor_sub).numpy()

    y_pred = y_pred_scaled * scaler_y_sub.scale_ + scaler_y_sub.mean_
    r2 = r2_score(y_sub[:,3], y_pred[:,3])
    r2_list.append(r2)

# # Plot
# plt.figure()
# plt.plot(sizes, r2_list, marker='o')
# plt.xlabel("Number of Data Points")
# plt.ylabel("R² Score")
# plt.title("R² vs Dataset Size")
# plt.grid()
# plt.show()

# -------------------------------
# 7️⃣ Full Training
# -------------------------------
epochs = 500

for epoch in range(epochs):
    optimizer.zero_grad()

    y_pred_scaled = model(X_tensor)
    y_pred = y_pred_scaled * y_scale + y_mean

    t_real = torch.tensor(X, dtype=torch.float32)
    t1, t2, t3 = t_real[:,0], t_real[:,1], t_real[:,2]

    E1_pred, E2_pred, E3_pred = y_pred[:,0], y_pred[:,1], y_pred[:,2]

    physics_loss = ((E1_pred - row_energy(t1))**2 +
                    (E2_pred - row_energy(t2))**2 +
                    (E3_pred - row_energy(t3))**2).mean()

    data_loss = loss_fn(y_pred_scaled, y_tensor)

    loss = data_weight * data_loss + physics_weight * physics_loss
    loss.backward()
    optimizer.step()

    if epoch % 500 == 0:
        print(f"Epoch {epoch} | Loss {loss.item():.6f}")

# -------------------------------
# 8️⃣ Predictions
# -------------------------------
with torch.no_grad():
    y_pred_scaled = model(X_tensor).numpy()

y_pred = y_pred_scaled * scaler_y.scale_ + scaler_y.mean_
E_total_pred = y_pred[:,3]

# -------------------------------
# 9️⃣ Extrapolation
# -------------------------------
N_extra = 5000

t1_extra = np.random.uniform(0.5, 3.5, N_extra)
t2_extra = np.random.uniform(0.5, 3.5, N_extra)
t3_extra = np.random.uniform(0.5, 3.5, N_extra)

X_extra = np.column_stack([t1_extra, t2_extra, t3_extra])
X_scaled_extra = scaler_X.transform(X_extra)
X_tensor_extra = torch.tensor(X_scaled_extra, dtype=torch.float32)

with torch.no_grad():
    y_extra_scaled = model(X_tensor_extra).numpy()

y_extra = y_extra_scaled * scaler_y.scale_ + scaler_y.mean_
E_total_extra = y_extra[:,3]

# -------------------------------
# 🔟 Equation reference (UPDATED)
# -------------------------------
E_total_eq_extra = (
    0.3 * np.exp(1.216 * t1_extra) +
    0.3 * np.exp(1.216 * t2_extra) +
    0.3 * np.exp(1.216 * t3_extra)
)

# -------------------------------
# 📊 Metrics
# -------------------------------
# MAE = mean_absolute_error(E_total_eq_extra, E_total_extra)
# RMSE = np.sqrt(mean_squared_error(E_total_eq_extra, E_total_extra))
# R2 = r2_score(E_total_eq_extra, E_total_extra)

# print("\nError vs Physics Equation:")
# print(f"MAE  : {MAE:.4f}")
# print(f"RMSE : {RMSE:.4f}")
# print(f"R²   : {R2:.4f}")

# -------------------------------
# 📈 Plots
# -------------------------------
E_total_data = data['e_total'].values

def plot_t_vs_E(t, name):
    plt.figure(figsize=(7,5))
    plt.scatter(t, E_total_eq_extra, color='blue', s=5, alpha=0.6, label="Equation")
    plt.scatter(t, E_total_extra, color='red', s=5, alpha=0.6, label="Predicted")
    plt.scatter(X[:,0 if name=='t1' else 1 if name=='t2' else 2],
                E_total_data, color='yellow', s=5, label="Dataset")

    plt.xlabel(f"{name} (mm)")
    plt.ylabel("Total Energy")
    plt.title(f"{name} vs E_total")
    plt.legend()
    plt.grid()
    plt.show()

plot_t_vs_E(t1_extra, 't1')
plot_t_vs_E(t2_extra, 't2')
plot_t_vs_E(t3_extra, 't3')

# -------------------------------
# Final comparison plot
# -------------------------------
# plt.figure(figsize=(6,6))
# plt.scatter(E_total_eq_extra, E_total_extra, alpha=0.6)

# plt.plot(
#     [E_total_eq_extra.min(), E_total_eq_extra.max()],
#     [E_total_eq_extra.min(), E_total_eq_extra.max()],
#     'r--'
# )

# plt.xlabel("Equation Energy")
# plt.ylabel("Predicted Energy")
# plt.title("Prediction vs Equation")
# plt.grid()
# plt.show()

In [ ]:
# PGNN inverse design inside data set########################################### report 9

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# =========================================
# 1️⃣ Load dataset
# =========================================

file_path = '/content/sample_data/data 2.xlsx'
data = pd.read_excel(file_path)

data.columns = data.columns.str.strip().str.lower()

e1 = data[['e1']].values.astype('float32')
e2 = data[['e2']].values.astype('float32')
e3 = data[['e3']].values.astype('float32')

t1 = data[['t1']].values.astype('float32')
t2 = data[['t2']].values.astype('float32')
t3 = data[['t3']].values.astype('float32')

E_total = (data['e1'] + data['e2'] + data['e3']).values

# =========================================
# 2️⃣ Normalization
# =========================================

scaler_e = StandardScaler()
scaler_t = StandardScaler()

e_all = np.vstack([e1,e2,e3])
t_all = np.vstack([t1,t2,t3])

scaler_e.fit(e_all)
scaler_t.fit(t_all)

e1_s = scaler_e.transform(e1)
e2_s = scaler_e.transform(e2)
e3_s = scaler_e.transform(e3)

t1_s = scaler_t.transform(t1)
t2_s = scaler_t.transform(t2)
t3_s = scaler_t.transform(t3)

# tensors
e1_t = torch.tensor(e1_s,dtype=torch.float32)
e2_t = torch.tensor(e2_s,dtype=torch.float32)
e3_t = torch.tensor(e3_s,dtype=torch.float32)

t1_t = torch.tensor(t1_s,dtype=torch.float32)
t2_t = torch.tensor(t2_s,dtype=torch.float32)
t3_t = torch.tensor(t3_s,dtype=torch.float32)

# =========================================
# 3️⃣ Neural Network
# =========================================

class Net(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(1,64),
            nn.ReLU(),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Linear(64,1)
        )

    def forward(self,x):

        return self.net(x)

# three independent models
model1 = Net()
model2 = Net()
model3 = Net()

opt1 = optim.Adam(model1.parameters(),lr=0.001)
opt2 = optim.Adam(model2.parameters(),lr=0.001)
opt3 = optim.Adam(model3.parameters(),lr=0.001)

loss_fn = nn.MSELoss()

# =========================================
# 4️⃣ Training
# =========================================

epochs = 1500

for epoch in range(epochs):

    # t1 model
    opt1.zero_grad()
    pred1 = model1(e1_t)
    loss1 = loss_fn(pred1,t1_t)
    loss1.backward()
    opt1.step()

    # t2 model
    opt2.zero_grad()
    pred2 = model2(e2_t)
    loss2 = loss_fn(pred2,t2_t)
    loss2.backward()
    opt2.step()

    # t3 model
    opt3.zero_grad()
    pred3 = model3(e3_t)
    loss3 = loss_fn(pred3,t3_t)
    loss3.backward()
    opt3.step()

    if epoch % 200 == 0:
        print(epoch, loss1.item(), loss2.item(), loss3.item())

# =========================================
# 5️⃣ Predictions
# =========================================

with torch.no_grad():

    t1_pred = scaler_t.inverse_transform(model1(e1_t).numpy())
    t2_pred = scaler_t.inverse_transform(model2(e2_t).numpy())
    t3_pred = scaler_t.inverse_transform(model3(e3_t).numpy())

# =========================================
# 6️⃣ Total Energy
# =========================================

E_total = e1.flatten() + e2.flatten() + e3.flatten()

# =========================================
# 7️⃣ Plot
# =========================================

# =========================================
# 7️⃣ Plot results
# =========================================

# =========================================
# 7️⃣ Plot Predicted vs Real
# =========================================

E_total = e1.flatten() + e2.flatten() + e3.flatten()

plt.figure(figsize=(15,4))

# -------- t1 --------
plt.subplot(1,3,1)
plt.scatter(E_total, t1_pred, color='blue', label='Predicted t1')
plt.scatter(E_total, t1.flatten(), color='orange', label='Real t1', alpha=0.6)
plt.xlabel("E_total")
plt.ylabel("t1")
plt.title("E_total vs t1")
plt.legend()
plt.grid(True)

# -------- t2 --------
plt.subplot(1,3,2)
plt.scatter(E_total, t2_pred, color='green', label='Predicted t2')
plt.scatter(E_total, t2.flatten(), color='orange', label='Real t2', alpha=0.6)
plt.xlabel("E_total")
plt.ylabel("t2")
plt.title("E_total vs t2")
plt.legend()
plt.grid(True)

# -------- t3 --------
plt.subplot(1,3,3)
plt.scatter(E_total, t3_pred, color='red', label='Predicted t3')
plt.scatter(E_total, t3.flatten(), color='orange', label='Real t3', alpha=0.6)
plt.xlabel("E_total")
plt.ylabel("t3")
plt.title("E_total vs t3")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# scateer graphes of PGNN inverse design e vs t report 10############

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# =========================================
# 1️⃣ Load dataset
# =========================================
file_path = "/content/sample_data/data_new.xlsx"

data = pd.read_excel(file_path)
data.columns = data.columns.str.strip().str.lower()

print("Dataset loaded from:", file_path)
print(data.head())

# =========================================
# 2️⃣ Extract data
# =========================================
e1 = data[['e1']].values.astype('float32')
e2 = data[['e2']].values.astype('float32')
e3 = data[['e3']].values.astype('float32')

t1 = data[['t1']].values.astype('float32')
t2 = data[['t2']].values.astype('float32')
t3 = data[['t3']].values.astype('float32')

# =========================================
# 3️⃣ Normalization
# =========================================
def normalize_data(x, y):
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_scaled = scaler_X.fit_transform(x)
    y_scaled = scaler_y.fit_transform(y)

    return (
        torch.tensor(X_scaled, dtype=torch.float32),
        torch.tensor(y_scaled, dtype=torch.float32),
        scaler_X,
        scaler_y
    )

# =========================================
# 4️⃣ Neural Network
# =========================================
class InverseNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

# =========================================
# 5️⃣ Physics equation (UPDATED)
# E = 0.4 * exp(0.95 * t)
# =========================================
def row_energy(t):
    return 0.4 * torch.exp(0.95 * t)

# =========================================
# 6️⃣ Train model
# =========================================
def train_model(e_vals, t_vals, physics_weight=0.05, epochs=1500):

    X_tensor, y_tensor, scaler_X, scaler_y = normalize_data(e_vals, t_vals)

    model = InverseNet()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    e_tensor = torch.tensor(e_vals, dtype=torch.float32).flatten()

    for epoch in range(epochs):
        optimizer.zero_grad()

        # NN prediction (scaled)
        t_scaled_pred = model(X_tensor)

        # Back to real scale
        t_pred = t_scaled_pred * torch.tensor(scaler_y.scale_) + torch.tensor(scaler_y.mean_)

        # Losses
        data_loss = loss_fn(t_scaled_pred, y_tensor)
        physics_loss = ((e_tensor - row_energy(t_pred.squeeze()))**2).mean()

        loss = data_loss + physics_weight * physics_loss

        loss.backward()
        optimizer.step()

        if epoch % 500 == 0:
            print(f"Epoch {epoch} | Loss {loss.item():.6f}")

    return model, scaler_X, scaler_y

# =========================================
# Train models
# =========================================
model1, scaler_X1, scaler_y1 = train_model(e1, t1)
model2, scaler_X2, scaler_y2 = train_model(e2, t2)
model3, scaler_X3, scaler_y3 = train_model(e3, t3)

# =========================================
# 7️⃣ Prediction
# =========================================
def predict(model, scaler_X, scaler_y, e_vals):
    X_scaled = scaler_X.transform(e_vals)
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

    t_scaled = model(X_tensor).detach().numpy()
    return scaler_y.inverse_transform(t_scaled)

# =========================================
# 8️⃣ Extrapolation
# =========================================
N_points = 10000

e1_extra = np.random.uniform(0.5, 18, N_points).reshape(-1,1)
e2_extra = np.random.uniform(0.5, 18, N_points).reshape(-1,1)
e3_extra = np.random.uniform(0.5, 18, N_points).reshape(-1,1)

t1_extra = predict(model1, scaler_X1, scaler_y1, e1_extra)
t2_extra = predict(model2, scaler_X2, scaler_y2, e2_extra)
t3_extra = predict(model3, scaler_X3, scaler_y3, e3_extra)

# Clamp thickness
t1_extra = np.clip(t1_extra, 0.5, 3.5)
t2_extra = np.clip(t2_extra, 0.5, 3.5)
t3_extra = np.clip(t3_extra, 0.5, 3.5)

# =========================================
# 9️⃣ Total energy (UPDATED)
# =========================================
E_total_extra = (
    0.4 * np.exp(0.95 * t1_extra[:,0]) +
    0.4 * np.exp(0.95 * t2_extra[:,0]) +
    0.4 * np.exp(0.95 * t3_extra[:,0])
)

# =========================================
# 🔟 Physics inverse (UPDATED)
# t = (1/0.95) * log(E / 0.4)
# =========================================
def safe_log(x):
    return np.log(np.maximum(x, 1e-8))

t1_eq_extra = np.clip((1/0.95) * safe_log(e1_extra / 0.4), 0.5, 3.5)
t2_eq_extra = np.clip((1/0.95) * safe_log(e2_extra / 0.4), 0.5, 3.5)
t3_eq_extra = np.clip((1/0.95) * safe_log(e3_extra / 0.4), 0.5, 3.5)

# =========================================
# 📊 Plots
# =========================================
def plot_results(E, t_nn, t_phys, label, color):
    plt.figure(figsize=(8,5))
    plt.scatter(E, t_nn, s=10, alpha=0.6, label=f'NN {label}', color=color)
    plt.scatter(E, t_phys, s=10, alpha=0.3, label='Physics', color='blue')
    plt.xlabel("Total Energy (J)")
    plt.ylabel(f"{label} (mm)")
    plt.title(f"{label} vs Energy")
    plt.grid(True)
    plt.legend()
    plt.show()

plot_results(E_total_extra, t1_extra, t1_eq_extra, "t1", "red")
plot_results(E_total_extra, t2_extra, t2_eq_extra, "t2", "green")
plot_results(E_total_extra, t3_extra, t3_eq_extra, "t3", "orange")

# =========================================
# 📈 R² Evaluation
# =========================================
r2_t1 = r2_score(t1_eq_extra.flatten(), t1_extra.flatten())
r2_t2 = r2_score(t2_eq_extra.flatten(), t2_extra.flatten())
r2_t3 = r2_score(t3_eq_extra.flatten(), t3_extra.flatten())

print(f"R² for t1: {r2_t1:.4f}")
print(f"R² for t2: {r2_t2:.4f}")
print(f"R² for t3: {r2_t3:.4f}")

In [ ]:
# #  graphes of PGNN inverse design errors report 14  ############


import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# =========================================
# 1️⃣ Load dataset
# =========================================
file_path = "/content/sample_data/data_new.xlsx"
data = pd.read_excel(file_path)
data.columns = data.columns.str.strip().str.lower()

print("Dataset loaded")

e = data[['e1']].values.astype('float32')
t = data[['t1']].values.astype('float32')


# =========================================
# 2️⃣ Physics model
# =========================================
def t_physics(E):
    return (1 / 0.95) * np.log(E / 0.4)


def E_from_t(t):
    return 0.4 * np.exp(0.95 * t)


# =========================================
# 3️⃣ Residual Neural Network
# =========================================
class ResidualNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 32),
            nn.Tanh(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)


# =========================================
# 4️⃣ Train model
# =========================================
def train_model(e_vals, t_vals, epochs=2000):

    e_tensor = torch.tensor(e_vals, dtype=torch.float32).flatten()
    t_tensor = torch.tensor(t_vals, dtype=torch.float32).flatten()

    model = ResidualNet()
    opt = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    for epoch in range(epochs):
        opt.zero_grad()

        e_in = e_tensor.unsqueeze(1)

        # physics baseline
        t_phys = torch.tensor(t_physics(e_tensor.numpy()), dtype=torch.float32)

        # NN correction
        delta = model(e_in).squeeze()

        t_pred = t_phys + delta

        loss = loss_fn(t_pred, t_tensor)

        loss.backward()
        opt.step()

        if epoch % 500 == 0:
            print(f"Epoch {epoch} Loss {loss.item():.6f}")

    return model


# =========================================
# 5️⃣ Train
# =========================================
model = train_model(e, t)


# =========================================
# 6️⃣ Prediction
# =========================================
def predict(model, E_vals):

    E_tensor = torch.tensor(E_vals, dtype=torch.float32).flatten()

    t_phys = t_physics(E_tensor.numpy())
    delta = model(E_tensor.unsqueeze(1)).detach().squeeze().numpy()

    t_pred = t_phys + delta

    # enforce physical bounds
    return np.clip(t_pred, 0.5, 3.5)


# =========================================
# 7️⃣ Extrapolation
# =========================================
N = 10000
E_extra = np.random.uniform(0.1, 18, N)

t_pred = predict(model, E_extra)
t_true = np.clip(t_physics(E_extra), 0.5, 3.5)


# =========================================
# 8️⃣ ERROR METRICS
# =========================================
abs_error = np.abs(t_pred - t_true)
rel_error = (abs_error / (np.abs(t_true) + 1e-8)) * 100


import numpy as np
import matplotlib.pyplot as plt

# =========================================
# 1️⃣ BINS
# =========================================
bins = np.linspace(0.5, 3.5, 20)
centers = 0.5 * (bins[:-1] + bins[1:])

abs_bin = []
rel_bin = []

# =========================================
# 2️⃣ BIN AVERAGING
# =========================================
for i in range(len(bins)-1):
    mask = (t_pred >= bins[i]) & (t_pred < bins[i+1])

    if np.sum(mask) > 0:
        abs_bin.append(np.mean(abs_error[mask]))
        rel_bin.append(np.mean(rel_error[mask]))
    else:
        abs_bin.append(np.nan)
        rel_bin.append(np.nan)

# =========================================
# 3️⃣ COLOR RULE
# =========================================
colors = []
for c in centers:
    if (0.5 <= c < 1.0) or (3.0 <= c <= 3.5):
        colors.append('red')   # boundary regions
    else:
        colors.append('blue')  # safe region


# =========================================
# 4️⃣ ABSOLUTE ERROR BAR PLOT
# =========================================
plt.figure(figsize=(9,5))

plt.bar(centers, abs_bin, width=0.15, color=colors, alpha=0.8)

plt.axvline(1, linestyle='--', color='black')
plt.axvline(3, linestyle='--', color='black')

plt.xlabel("Thickness (mm)")
plt.ylabel("Absolute Error (mm)")
plt.title("Absolute Error (Red = 0.5–1 & 3–3.5)")
plt.grid(True)
plt.show()


# =========================================
# 5️⃣ RELATIVE ERROR BAR PLOT (%)
# =========================================
plt.figure(figsize=(9,5))

plt.bar(centers, rel_bin, width=0.15, color=colors, alpha=0.8)

plt.axvline(1, linestyle='--', color='black')
plt.axvline(3, linestyle='--', color='black')

plt.xlabel("Thickness (mm)")
plt.ylabel("Relative Error (%)")
plt.title("Relative Error (Red = 0.5–1 & 3–3.5)")
plt.grid(True)
plt.show()

In [ ]:
# scateer graphes of PGNN inverse design e vs t report 14 extrapolation highlit ############


import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# =========================================
# 🔥 HIGH RESOLUTION SETTINGS
# =========================================
plt.rcParams['figure.dpi'] = 200
plt.rcParams['savefig.dpi'] = 300


# =========================================
# 1️⃣ LOAD DATA
# =========================================
file_path = "/content/sample_data/data_new.xlsx"
data = pd.read_excel(file_path)
data.columns = data.columns.str.strip().str.lower()

e1 = data[['e1']].values.astype('float32')
e2 = data[['e2']].values.astype('float32')
e3 = data[['e3']].values.astype('float32')

t1 = data[['t1']].values.astype('float32')
t2 = data[['t2']].values.astype('float32')
t3 = data[['t3']].values.astype('float32')


# =========================================
# 2️⃣ EXPONENTIAL PHYSICS
# =========================================
def t_physics(E):
    return (1 / 0.95) * np.log(E / 0.4)

def E_from_t(t):
    return 0.4 * np.exp(0.95 * t)


# =========================================
# 3️⃣ RESIDUAL NEURAL NETWORK
# =========================================
class ResidualNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)


# =========================================
# 4️⃣ TRAIN MODEL
# =========================================
def train_model(e_vals, t_vals, physics_weight=20.0, epochs=2000):

    e = torch.tensor(e_vals, dtype=torch.float32).flatten()
    t = torch.tensor(t_vals, dtype=torch.float32).flatten()

    model = ResidualNet()
    opt = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    for epoch in range(epochs):
        opt.zero_grad()

        t_phys = torch.tensor(t_physics(e.numpy()), dtype=torch.float32)
        delta = model(e.unsqueeze(1)).squeeze()

        t_pred = t_phys + delta

        data_loss = loss_fn(t_pred, t)

        e_recon = torch.tensor(E_from_t(t_pred.detach().numpy()), dtype=torch.float32)
        physics_loss = ((e - e_recon) ** 2).mean()

        loss = data_loss + physics_weight * physics_loss

        loss.backward()
        opt.step()

        if epoch % 500 == 0:
            print(f"Epoch {epoch} Loss {loss.item():.6f}")

    return model


# =========================================
# 5️⃣ TRAIN MODELS
# =========================================
model1 = train_model(e1, t1)
model2 = train_model(e2, t2)
model3 = train_model(e3, t3)


# =========================================
# 6️⃣ PREDICTION
# =========================================
def predict(model, e_vals):
    e = torch.tensor(e_vals, dtype=torch.float32).flatten()

    t_phys = t_physics(e.numpy())
    delta = model(e.unsqueeze(1)).detach().squeeze().numpy()

    return t_phys + delta


# =========================================
# 7️⃣ EXTRAPOLATION
# =========================================
N = 10000

e1_extra = np.random.uniform(0.5, 18, N).reshape(-1,1)
e2_extra = np.random.uniform(0.5, 18, N).reshape(-1,1)
e3_extra = np.random.uniform(0.5, 18, N).reshape(-1,1)

t1_extra = predict(model1, e1_extra)
t2_extra = predict(model2, e2_extra)
t3_extra = predict(model3, e3_extra)


# Physics reference
t1_eq = t_physics(e1_extra.flatten())
t2_eq = t_physics(e2_extra.flatten())
t3_eq = t_physics(e3_extra.flatten())


# =========================================
# 8️⃣ ENERGY CONSISTENCY
# =========================================
E_total = (
    0.4 * np.exp(0.95 * t1_extra) +
    0.4 * np.exp(0.95 * t2_extra) +
    0.4 * np.exp(0.95 * t3_extra)
)


# =========================================
# 9️⃣ FILTER RANGE
# =========================================
mask1 = (t1_extra >= 0.5) & (t1_extra <= 3.5)
mask2 = (t2_extra >= 0.5) & (t2_extra <= 3.5)
mask3 = (t3_extra >= 0.5) & (t3_extra <= 3.5)


# =========================================
# 🔟 HIGH-RES PLOTS
# =========================================

def save_plot(name):
    plt.savefig(name, bbox_inches='tight', dpi=300)


# --- t1 ---
plt.figure(figsize=(8,5), dpi=200)
plt.scatter(E_total[mask1], t1_extra[mask1], s=10, color='gold', alpha=0.6, label='NN+Physics t1')
plt.scatter(E_total[mask1], t1_eq[mask1], s=10, color='blue', alpha=0.3, label='Physics t1')
plt.xlabel("Energy")
plt.ylabel("t1")
plt.title("t1 vs E (High Resolution)")
plt.grid()
plt.legend()
save_plot("t1_plot.png")
plt.show()


# --- t2 ---
plt.figure(figsize=(8,5), dpi=200)
plt.scatter(E_total[mask2], t2_extra[mask2], s=10, color='gold', alpha=0.6, label='NN+Physics t2')
plt.scatter(E_total[mask2], t2_eq[mask2], s=10, color='green', alpha=0.3, label='Physics t2')
plt.xlabel("Energy")
plt.ylabel("t2")
plt.title("t2 vs E (High Resolution)")
plt.grid()
plt.legend()
save_plot("t2_plot.png")
plt.show()


# --- t3 ---
plt.figure(figsize=(8,5), dpi=200)
plt.scatter(E_total[mask3], t3_extra[mask3], s=10, color='gold', alpha=0.6, label='NN+Physics t3')
plt.scatter(E_total[mask3], t3_eq[mask3], s=10, color='red', alpha=0.3, label='Physics t3')
plt.xlabel("Energy")
plt.ylabel("t3")
plt.title("t3 vs E (High Resolution)")
plt.grid()
plt.legend()
save_plot("t3_plot.png")
plt.show()